# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Author:** Swaraj Kumar

**Document:** Regulation (EU) 2016/679 – General Data Protection Regulation (GDPR), 88 pages.
It is a long legal text full of general rules, exceptions ("shall not apply where…") and prohibitions,
which makes it a good stress test for retrieval.

**Pipelines compared over the exact same chunk corpus**

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder retrieves Top-10 by cosine similarity | same Top-10 candidates |
| Stage 2 | – | Cross-encoder scores all 10 (query, chunk) pairs, re-sorts |
| LLM context | Top-3 from bi-encoder | Top-3 after re-ranking |
| Prompt | identical | identical |

## 1. Setup & configuration

In [1]:
import hashlib
import json
import os
import platform
import re
import time
from dataclasses import dataclass, field
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"  # harmless Windows cache warning

import numpy as np
import pandas as pd
import pymupdf
from IPython.display import display
from sentence_transformers import CrossEncoder, SentenceTransformer

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

In [2]:
PDF_PATH = Path("data/gdpr.pdf")

BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

LLM_MODEL = "gemma-4-31b-it"  # Google open-weights model, served through the Gemini API
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")  # set this before starting Jupyter

USE_LLM_CACHE = True   # False = always call the LLM again instead of reusing data/llm_answers.json

MAX_CHUNK_WORDS = 180  # upper bound on chunk length (excluding the section label)
RETRIEVE_K = 10        # candidates returned by the bi-encoder
CONTEXT_K = 3          # chunks passed to the LLM

## 2. Load and clean the PDF

Every page ends with the Official Journal footer (date, `L 119/NN`, journal name, `EN`) and some pages carry
legal footnotes such as `(1) OJ C 229, …`. Both are noise for retrieval, so they are removed here.
Each remaining line keeps its page number so that chunks can be traced back to the source.

In [3]:
FOOTER_LINE = re.compile(r"^(\d{1,2}\.\d{1,2}\.\d{4}|L \d+/\d+|Official Journal of the European Union|EN)$")
FOOTNOTE_START = re.compile(r"^\(\d\) (Regulation|Directive|Decision|Commission|Council|Position|OJ)")


def read_pdf_lines(path):
    """Return a list of (page_number, line_text) with page footers removed."""
    lines = []
    for page in pymupdf.open(path):
        for line in page.get_text().splitlines():
            line = line.strip()
            if line and not FOOTER_LINE.match(line):
                lines.append((page.number + 1, line))
    return lines


pdf_lines = read_pdf_lines(PDF_PATH)
print(f"{len(pdf_lines)} lines of text extracted")

4187 lines of text extracted


## 3. Structure-aware, sentence-aware chunking

The GDPR has two parts:

* **Recitals (1)–(173)** – explanatory paragraphs before the words *"HAVE ADOPTED THIS REGULATION"*.
* **Articles 1–99** – the binding rules, each starting with `Article N` followed by its title.

Chunking rules:

1. Chunks never cross a recital/article boundary.
2. Articles are split on their numbered paragraphs (`1.`, `2.`, …).
3. Paragraphs are split into sentences (or `;`-separated list items) and **whole sentences** are packed into
   chunks of at most `MAX_CHUNK_WORDS` words, so no chunk starts or ends in the middle of a sentence.
4. When a chunk boundary falls inside a paragraph, the last sentence is repeated at the start of the next chunk
   (one-sentence overlap), so a rule and its condition are not separated without context.
5. Each chunk is prefixed with its section label (e.g. `[Article 34 - Communication of a personal data breach …]`)
   so both encoders know where the text comes from.

In [4]:
@dataclass
class Chunk:
    chunk_id: int
    section: str
    page: int
    text: str


def is_recital_header(match, last_recital):
    """Recitals are numbered in order; (1)-(99) stand alone on a line, (100)+ share it with text."""
    number, rest = int(match.group(1)), match.group(2)
    if number != last_recital + 1:
        return False
    return number >= 100 or not rest


def split_into_sections(lines):
    """Group lines into recitals and articles."""
    sections = []
    current = None
    in_articles = False
    last_recital = 0
    footnote_page = None
    i = 0

    while i < len(lines):
        page, line = lines[i]
        recital = re.fullmatch(r"\((\d{1,3})\)\s*(.*)", line)
        article = re.fullmatch(r"Article (\d{1,2})", line)

        if line.startswith("HAVE ADOPTED THIS REGULATION"):
            in_articles = True
        elif FOOTNOTE_START.match(line):
            footnote_page = page  # footnotes run to the bottom of the page
        elif page == footnote_page:
            pass
        elif not in_articles and recital and is_recital_header(recital, last_recital):
            last_recital += 1
            first_line = [recital.group(2)] if recital.group(2) else []
            current = {"section": f"Recital {last_recital}", "page": page, "lines": first_line}
            sections.append(current)
        elif in_articles and article:
            title = lines[i + 1][1]
            current = {"section": f"Article {article.group(1)} - {title}", "page": page, "lines": []}
            sections.append(current)
            i += 1  # the title line is consumed
        elif in_articles and re.fullmatch(r"(CHAPTER|Section) [IVX\d]+", line):
            i += 1  # skip chapter/section headings and their titles
        elif current is not None:
            current["lines"].append(line)
        i += 1

    return sections


def split_paragraphs(section_lines):
    """Split an article on its numbered paragraphs ("1.", "2.", ...)."""
    paragraphs, buffer = [], []
    for line in section_lines:
        if re.fullmatch(r"\d{1,2}\.", line) and buffer:
            paragraphs.append(" ".join(buffer))
            buffer = []
        buffer.append(line)
    if buffer:
        paragraphs.append(" ".join(buffer))
    return paragraphs


SENTENCE_BOUNDARY = re.compile(r"(?<=[.;])\s+(?=[A-Z(])")


def split_sentences(paragraph, max_words):
    """Split a paragraph into sentences / list items; cut any unit longer than max_words."""
    units = []
    for sentence in SENTENCE_BOUNDARY.split(paragraph):
        words = sentence.split()
        for start in range(0, len(words), max_words):
            units.append(" ".join(words[start:start + max_words]))
    return units


def word_count(units):
    return sum(len(unit.split()) for unit in units)


def pack_sentences(paragraphs, max_words, max_overlap_words=60):
    """Greedily pack whole sentences into chunks of at most max_words words.

    If a chunk boundary falls inside a paragraph, the previous sentence is carried over
    into the next chunk (one-sentence overlap).
    """
    chunks, current = [], []
    for paragraph in paragraphs:
        for position, sentence in enumerate(split_sentences(paragraph, max_words)):
            if current and word_count(current + [sentence]) > max_words:
                chunks.append(" ".join(current))
                overlap = current[-1]
                keep_overlap = (position > 0
                                and len(overlap.split()) <= max_overlap_words
                                and word_count([overlap, sentence]) <= max_words)
                current = [overlap] if keep_overlap else []
            current.append(sentence)
    if current:
        chunks.append(" ".join(current))
    return chunks


def build_chunks(lines, max_words=MAX_CHUNK_WORDS):
    chunks = []
    for section in split_into_sections(lines):
        for body in pack_sentences(split_paragraphs(section["lines"]), max_words):
            text = f"[{section['section']}] {body}"
            chunks.append(Chunk(len(chunks), section["section"], section["page"], text))
    return chunks


chunks = build_chunks(pdf_lines)
chunk_texts = [c.text for c in chunks]

word_counts = [len(t.split()) for t in chunk_texts]
n_articles = len({c.section for c in chunks if c.section.startswith("Article")})
n_recitals = len({c.section for c in chunks if c.section.startswith("Recital")})
print(f"{len(chunks)} chunks | {n_recitals} recitals | {n_articles} articles")
bodies = [t.split("] ", 1)[1] for t in chunk_texts]
cut_mid_sentence = sum(not re.search(r"[.;:)]$", body.strip()) for body in bodies)
print(f"words per chunk (incl. label): mean={np.mean(word_counts):.0f}, max={max(word_counts)}")
print(f"chunks ending mid-sentence: {cut_mid_sentence}")

471 chunks | 173 recitals | 99 articles
words per chunk (incl. label): mean=125, max=193
chunks ending mid-sentence: 4


In [5]:
# Example: Article 34 - the general rule (para 1-2) and its exceptions (para 3) end up in different chunks
for c in chunks:
    if c.section.startswith("Article 34 "):
        print(f"chunk {c.chunk_id} (page {c.page}):\n{c.text}\n")

chunk 315 (page 52):
[Article 34 - Communication of a personal data breach to the data subject] 1. When the personal data breach is likely to result in a high risk to the rights and freedoms of natural persons, the controller shall communicate the personal data breach to the data subject without undue delay. 2. The communication to the data subject referred to in paragraph 1 of this Article shall describe in clear and plain language the nature of the personal data breach and contain at least the information and measures referred to in points (b), (c) and (d) of Article 33(3). 3. The communication to the data subject referred to in paragraph 1 shall not be required if any of the following conditions are met: (a) the controller has implemented appropriate technical and organisational protection measures, and those measures were applied to the personal data affected by the personal data breach, in particular those that render the personal data unintelligible to any person who is not autho

## 4. Vector store with a Bi-Encoder (Pipeline 1, step a)

The bi-encoder embeds every chunk **independently** into a 384-dim vector. Embeddings are L2-normalised,
so the inner product equals cosine similarity. With a few hundred chunks a NumPy matrix is a perfectly good
exact vector store (a FAISS `IndexFlatIP` would compute exactly the same thing).

In [6]:
class VectorStore:
    """Minimal exact-search vector store using normalised embeddings and inner product."""

    def __init__(self, model):
        self.model = model
        self.embeddings = None

    def index(self, texts):
        self.embeddings = self.model.encode(texts, batch_size=32, normalize_embeddings=True)

    def search(self, query, k):
        """Return [(chunk_index, cosine_score)] for the k most similar chunks."""
        query_vec = self.model.encode(query, normalize_embeddings=True)
        scores = self.embeddings @ query_vec
        top = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in top]


bi_encoder = SentenceTransformer(BI_ENCODER_NAME)
store = VectorStore(bi_encoder)

start = time.perf_counter()
store.index(chunk_texts)
index_seconds = time.perf_counter() - start
print(f"Indexed {len(chunk_texts)} chunks -> matrix {store.embeddings.shape} in {index_seconds:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 471 chunks -> matrix (471, 384) in 9.22s


## 5. Cross-Encoder (Pipeline 2, step b)

The cross-encoder reads the query and the chunk **together** in one transformer pass and outputs a single
relevance score. It cannot pre-compute anything, so it is only applied to the 10 candidates.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 6. LLM and the shared prompt

Both pipelines use **exactly the same prompt template**; only the three context chunks differ.
Temperature is 0 so differences in the answers come from the context, not from sampling.
The LLM is **Gemma 4 31B** (`gemma-4-31b-it`), Google's open-weights model, called through the Gemini API.
Generated answers are cached in `data/llm_answers.json`, so re-running the notebook reuses them
(and works without an API key). Set `USE_LLM_CACHE = False` to regenerate them. If no key and no cached answer exist, the answer cells show a placeholder.

In [8]:
PROMPT_TEMPLATE = """You are a legal assistant answering questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the context below. Cite the Article or Recital you rely on.
If the context does not contain the answer, say "The provided context does not answer this."

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question, hits):
    context = "\n\n".join(f"({i}) {hit.chunk.text}" for i, hit in enumerate(hits, start=1))
    return PROMPT_TEMPLATE.format(context=context, question=question)


llm_client = None
if GEMINI_API_KEY:
    from google import genai
    from google.genai import errors, types

    llm_client = genai.Client(api_key=GEMINI_API_KEY)
    llm_config = types.GenerateContentConfig(temperature=0.0)
else:
    print("GEMINI_API_KEY not set - using cached answers from data/llm_answers.json only.")


# Answers are cached on disk so re-running the notebook does not spend API quota again.
LLM_CACHE_PATH = Path("data/llm_answers.json")
llm_cache = json.loads(LLM_CACHE_PATH.read_text(encoding="utf-8")) if LLM_CACHE_PATH.exists() else {}

RETRYABLE_STATUS = {429, 500, 503}  # rate limit / temporary server errors


def call_llm(prompt, max_retries=6):
    """Call Gemini once, waiting and retrying on rate limits or temporary outages."""
    for attempt in range(max_retries):
        try:
            start = time.perf_counter()
            response = llm_client.models.generate_content(model=LLM_MODEL, contents=prompt, config=llm_config)
            return response.text.strip(), (time.perf_counter() - start) * 1000
        except errors.APIError as error:
            if error.code not in RETRYABLE_STATUS or attempt == max_retries - 1:
                raise
            time.sleep(30)


def generate_answer(prompt):
    """Return (answer_text, latency_ms of the API call)."""
    key = hashlib.sha256(f"{LLM_MODEL}|{prompt}".encode()).hexdigest()
    if USE_LLM_CACHE and key in llm_cache:
        return llm_cache[key]["answer"], llm_cache[key]["latency_ms"]
    if llm_client is None:
        return "[LLM skipped - set GEMINI_API_KEY to generate answers]", None

    answer, latency_ms = call_llm(prompt)
    llm_cache[key] = {"answer": answer, "latency_ms": latency_ms}
    LLM_CACHE_PATH.write_text(json.dumps(llm_cache, indent=2), encoding="utf-8")
    return answer, latency_ms

GEMINI_API_KEY not set - using cached answers from data/llm_answers.json only.


## 7. The two retrieval pipelines

In [9]:
@dataclass
class Hit:
    chunk: Chunk
    baseline_rank: int          # 1-based rank from the bi-encoder
    bi_score: float             # cosine similarity
    rerank_rank: int = None     # 1-based rank after the cross-encoder
    ce_score: float = None      # cross-encoder relevance logit


def baseline_retrieve(query, k=RETRIEVE_K):
    """Pipeline 1 (a-b): Top-k chunks by cosine similarity."""
    results = store.search(query, k)
    return [Hit(chunks[idx], rank, score) for rank, (idx, score) in enumerate(results, start=1)]


def rerank(query, candidates):
    """Pipeline 2 (b-c): score every (query, chunk) pair and sort by cross-encoder score."""
    scores = cross_encoder.predict([(query, hit.chunk.text) for hit in candidates])
    for hit, score in zip(candidates, scores):
        hit.ce_score = float(score)
    reranked = sorted(candidates, key=lambda hit: hit.ce_score, reverse=True)
    for rank, hit in enumerate(reranked, start=1):
        hit.rerank_rank = rank
    return reranked


@dataclass
class PipelineRun:
    query: str
    baseline_hits: list           # Top-10 in bi-encoder order
    reranked_hits: list           # same 10 in cross-encoder order
    retrieval_ms: float           # bi-encoder search time
    rerank_ms: float              # cross-encoder time
    answer_1: str = None
    answer_2: str = None
    llm_1_ms: float = None
    llm_2_ms: float = None


def run_pipelines(query, with_llm=True):
    """Run both pipelines on the same Top-10 candidate set."""
    t0 = time.perf_counter()
    candidates = baseline_retrieve(query)
    t1 = time.perf_counter()
    reranked = rerank(query, list(candidates))
    t2 = time.perf_counter()

    run = PipelineRun(query, candidates, reranked, (t1 - t0) * 1000, (t2 - t1) * 1000)

    if with_llm:
        run.answer_1, run.llm_1_ms = generate_answer(build_prompt(query, candidates[:CONTEXT_K]))
        run.answer_2, run.llm_2_ms = generate_answer(build_prompt(query, reranked[:CONTEXT_K]))
    return run

## 8. Evaluation and failure-mode stress tests

Each test case lists **evidence phrases** taken verbatim from the GDPR text that a correct answer needs.
A chunk counts as the *critical chunk* if it contains one of these phrases. This makes the rank shifts
measurable instead of eyeballed.

* **8.1** runs both pipelines on **56 questions** (`data/eval_queries.json`) to measure the overall effect.
* **8.2 / 8.3** zoom in on 9 showcase cases chosen from those 56: 7 where re-ranking rescues a buried chunk and 2 counter-examples where it does not.

In [10]:
@dataclass
class TestCase:
    name: str
    category: str
    query: str
    evidence: list = field(default_factory=list)
    why_hard: str = ""


TEST_CASES = [
    TestCase(
        name='Data portability vs. international transfers',
        category='Keyword trap',
        query='Can I request a transfer of my personal data to another controller?',
        evidence=['right to have the personal data transmitted directly from one controller to another'],
        why_hard="'transfer' is the key term of Chapter V (transfers to third countries) and of recitals on intra-group transfers; the real answer (Art. 20, data portability) says 'transmitted' instead.",
    ),
    TestCase(
        name='Sub-processor liability',
        category='Keyword trap',
        query='Who is liable if a sub-processor breaches its obligations?',
        evidence=['the initial processor shall remain fully liable to the controller'],
        why_hard="'liable' and 'damage' pull in the general compensation rule (Art. 82, Recital 146); the specific sub-processor rule is a single sentence in Art. 28(4).",
    ),
    TestCase(
        name='Telling individuals about a breach',
        category='Exception / corner case',
        query='Do I always have to tell affected individuals about a data breach?',
        evidence=['shall not be required if any of the following conditions'],
        why_hard='Art. 34(3) lists three exceptions, but the list is split over two chunks: (a) encryption is in the chunk with the general duty, (b) later measures and (c) disproportionate effort are in the next chunk. Each pipeline only sees part of the list.',
    ),
    TestCase(
        name='Dismissal of the DPO',
        category='Negation / role limit',
        query='Can a data protection officer be fired for doing their job?',
        evidence=['shall not be dismissed or penalised'],
        why_hard="Articles 37, 38 and 39 are all about the DPO; only one chunk contains the prohibition 'shall not be dismissed or penalised', and the word 'fired' never appears in the text.",
    ),
    TestCase(
        name='Police and criminal investigations',
        category='Document-specific: scope exclusion',
        query='Does GDPR apply to the police investigating crimes?',
        evidence=['by competent authorities for the purposes of the prevention, investigation'],
        why_hard='Chunks about criminal-conviction data (Art. 10) look relevant, but the answer is the scope exclusion in Art. 2(2)(d) / Recital 19 (that processing falls under Directive 2016/680).',
    ),
    TestCase(
        name='Processor must alert the controller',
        category='Document-specific: role-specific duty',
        query='Must a processor tell the controller about a breach?',
        evidence=['The processor shall notify the controller without undue delay'],
        why_hard='Chunks about processors in general (Art. 28, 29) outrank the one sentence in Art. 33(2) that gives the processor its own breach duty.',
    ),
    TestCase(
        name='Secrecy of the supervisory authority',
        category='Buried chunk (rank 10)',
        query='Does the regulator have to keep secrets?',
        evidence=['duty of professional secrecy'],
        why_hard="Art. 90 'Obligations of secrecy' (about professional secrecy of controllers) is the obvious lexical match; the duty of the authority's own staff is in Art. 54(2), ranked last by the bi-encoder.",
    ),
    TestCase(
        name='Regulator ignores a complaint',
        category='Limitation: re-ranking hurts',
        query='What can I do if the regulator ignores my complaint?',
        evidence=['does not handle a complaint or does not inform the data subject within three months'],
        why_hard='Counter-example: the cross-encoder prefers recitals that talk about complaints in general and pushes the Art. 78(2) remedy further down.',
    ),
    TestCase(
        name='Switching provider',
        category='Limitation: recall ceiling',
        query='Can I transfer my data to another company that offers the same service?',
        evidence=['right to have the personal data transmitted directly from one controller to another'],
        why_hard='Counter-example: the keyword trap is so strong that Art. 20 is not in the bi-encoder Top-10 at all, so the re-ranker never sees it. Re-ranking can only reorder what stage 1 retrieved.',
    ),
]


def critical_rank(hits, evidence, rank_attr):
    """Rank of the first hit that contains any evidence phrase (None if not in the Top-10)."""
    for hit in hits:
        if any(phrase.lower() in hit.chunk.text.lower() for phrase in evidence):
            return getattr(hit, rank_attr)
    return None


def is_critical(hit, evidence):
    return any(phrase.lower() in hit.chunk.text.lower() for phrase in evidence)

### 8.1 Aggregate evaluation over 56 questions

For every question the *critical chunk* is located in the bi-encoder Top-10 and again after re-ranking.

* **Hit@k** – share of questions whose critical chunk is in the Top-k (Hit@3 = it reaches the LLM).
* **MRR@10** – mean of 1/rank (0 when the chunk is not in the Top-10).
* **Hit@10** is identical for both pipelines by construction: re-ranking only reorders the same 10 candidates.

In [11]:
EVAL_QUERIES = json.loads(Path("data/eval_queries.json").read_text(encoding="utf-8"))

eval_rows = []
for item in EVAL_QUERIES:
    run = run_pipelines(item["query"], with_llm=False)
    eval_rows.append({
        "query": item["query"],
        "baseline rank": critical_rank(run.baseline_hits, item["evidence"], "baseline_rank"),
        "re-ranked rank": critical_rank(run.reranked_hits, item["evidence"], "rerank_rank"),
    })

evaluation = pd.DataFrame(eval_rows)
baseline_ranks = evaluation["baseline rank"].astype(float)
reranked_ranks = evaluation["re-ranked rank"].astype(float)


def hit_at(ranks, k):
    return (ranks <= k).mean()


def mrr(ranks):
    return (1 / ranks).fillna(0).mean()


metrics = pd.DataFrame({
    "Pipeline 1 (bi-encoder)": [hit_at(baseline_ranks, k) for k in (1, 3, 10)] + [mrr(baseline_ranks)],
    "Pipeline 2 (re-ranked)": [hit_at(reranked_ranks, k) for k in (1, 3, 10)] + [mrr(reranked_ranks)],
}, index=["Hit@1", "Hit@3 (reaches the LLM)", "Hit@10", "MRR@10"])

print(f"{len(evaluation)} questions")
display(metrics.round(3))

in_top3_before = baseline_ranks <= CONTEXT_K
in_top3_after = reranked_ranks <= CONTEXT_K
print(f"Critical chunk pulled INTO the Top-3 by re-ranking : {(~in_top3_before & in_top3_after).sum()}")
print(f"Critical chunk pushed OUT of the Top-3 by re-ranking: {(in_top3_before & ~in_top3_after).sum()}")
print(f"Critical chunk not in the Top-10 (cannot be fixed)  : {baseline_ranks.isna().sum()}")

56 questions


,Pipeline 1 (bi-encoder),Pipeline 2 (re-ranked)
Hit@1,0.339,0.464
Hit@3 (reaches the LLM),0.589,0.661
Hit@10,0.804,0.804
MRR@10,0.484,0.583


Critical chunk pulled INTO the Top-3 by re-ranking : 7
Critical chunk pushed OUT of the Top-3 by re-ranking: 3
Critical chunk not in the Top-10 (cannot be fixed)  : 11


In [12]:
def outcome(row):
    before, after = row["baseline rank"], row["re-ranked rank"]
    if pd.isna(before):
        return "not retrieved"
    if before > CONTEXT_K and after <= CONTEXT_K:
        return "rescued"
    if before <= CONTEXT_K and after > CONTEXT_K:
        return "lost"
    return "better" if after < before else ("worse" if after > before else "same")


evaluation["outcome"] = evaluation.apply(outcome, axis=1)
evaluation.astype({"baseline rank": "Int64", "re-ranked rank": "Int64"})

,query,baseline rank,re-ranked rank,outcome
0,Do I always have to tell affected individuals about a data breach?,5,2,rescued
1,How quickly must a company report a hack to the regulator?,2,1,better
2,Is processing health data prohibited and what are the exceptions?,<NA>,<NA>,not retrieved
3,At what age can a child consent to an app on their own?,1,1,same
4,What is the maximum fine for breaking the core data processing principles?,3,1,better
5,Can a bank reject my loan by computer alone without a human?,<NA>,<NA>,not retrieved
6,Does the regulation apply to my personal address book at home?,1,1,same
7,When is a company not required to appoint a data protection officer?,3,4,lost
8,Can a company charge me money for giving me a copy of my data?,1,1,same
9,Can the right to be forgotten be refused?,1,3,worse


### 8.2 Rank-shift summary of the showcase cases

Retrieval only (no LLM yet). `baseline rank` is the position of the critical chunk in the bi-encoder Top-10,
`re-ranked rank` its position after the cross-encoder. Anything > 3 is **not seen by the LLM**.

In [13]:
runs = {case.name: run_pipelines(case.query, with_llm=False) for case in TEST_CASES}


def fmt_rank(rank):
    return f"#{rank}" if rank else "not in Top-10"


summary_rows = []
for case in TEST_CASES:
    run = runs[case.name]
    before = critical_rank(run.baseline_hits, case.evidence, "baseline_rank")
    after = critical_rank(run.reranked_hits, case.evidence, "rerank_rank")
    critical = next((h for h in run.reranked_hits if is_critical(h, case.evidence)), None)
    summary_rows.append({
        "case": case.name,
        "category": case.category,
        "critical chunk": f"{critical.chunk.chunk_id} ({critical.chunk.section.split(' - ')[0]})" if critical else "-",
        "baseline rank": fmt_rank(before),
        "re-ranked rank": fmt_rank(after),
        "in LLM context (P1 / P2)": f"{'yes' if before and before <= CONTEXT_K else 'NO'} / "
                                    f"{'yes' if after and after <= CONTEXT_K else 'NO'}",
    })

pd.DataFrame(summary_rows).set_index("case")

,category,critical chunk,baseline rank,re-ranked rank,in LLM context (P1 / P2)
case,,,,,
Data portability vs. international transfers,Keyword trap,285 (Article 20),#4,#1,NO / yes
Sub-processor liability,Keyword trap,304 (Article 28),#4,#1,NO / yes
Telling individuals about a breach,Exception / corner case,315 (Article 34),#5,#2,NO / yes
Dismissal of the DPO,Negation / role limit,327 (Article 38),#7,#2,NO / yes
Police and criminal investigations,Document-specific: scope exclusion,20 (Recital 19),#4,#1,NO / yes
Processor must alert the controller,Document-specific: role-specific duty,313 (Article 33),#6,#1,NO / yes
Secrecy of the supervisory authority,Buried chunk (rank 10),375 (Article 54),#10,#2,NO / yes
Regulator ignores a complaint,Limitation: re-ranking hurts,436 (Article 78),#4,#8,NO / NO
Switching provider,Limitation: recall ceiling,-,not in Top-10,not in Top-10,NO / NO


### 8.3 Detailed view per showcase case

For each case: the full Top-10 with both scores, then Answer 1 (baseline context) vs Answer 2 (re-ranked context).
`*` marks the critical chunk.

In [14]:
def rank_table(run, case):
    rows = []
    for hit in run.baseline_hits:
        rows.append({
            "chunk": hit.chunk.chunk_id,
            "section": hit.chunk.section[:60],
            "critical": "*" if is_critical(hit, case.evidence) else "",
            "bi-encoder rank": hit.baseline_rank,
            "cosine": round(hit.bi_score, 3),
            "cross-encoder rank": hit.rerank_rank,
            "CE score": round(hit.ce_score, 2),
            "shift": hit.baseline_rank - hit.rerank_rank,
        })
    return pd.DataFrame(rows).set_index("chunk").sort_values("cross-encoder rank")


def show_case(case):
    run = run_pipelines(case.query, with_llm=True)
    runs[case.name] = run
    before = critical_rank(run.baseline_hits, case.evidence, "baseline_rank")
    after = critical_rank(run.reranked_hits, case.evidence, "rerank_rank")

    print("=" * 110)
    print(f"{case.name}  [{case.category}]")
    print(f"Query: {case.query}")
    print(f"Why it is hard: {case.why_hard}")
    print(f"Critical chunk: {fmt_rank(before)} -> {fmt_rank(after)}")
    display(rank_table(run, case))
    print("--- Answer 1 (baseline Top-3:", [h.chunk.chunk_id for h in run.baseline_hits[:CONTEXT_K]], ")")
    print(run.answer_1)
    print()
    print("--- Answer 2 (re-ranked Top-3:", [h.chunk.chunk_id for h in run.reranked_hits[:CONTEXT_K]], ")")
    print(run.answer_2)

#### Case 1: Data portability vs. international transfers (Keyword trap)

In [15]:
show_case(TEST_CASES[0])

Data portability vs. international transfers  [Keyword trap]
Query: Can I request a transfer of my personal data to another controller?
Why it is hard: 'transfer' is the key term of Chapter V (transfers to third countries) and of recitals on intra-group transfers; the real answer (Art. 20, data portability) says 'transmitted' instead.
Critical chunk: #4 -> #1


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
285,Article 20 - Right to data portability,*,5,0.531,1,4.61,4
88,Recital 68,,9,0.515,2,3.93,7
276,Article 15 - Right of access by the data subject,,10,0.514,3,3.32,7
82,Recital 63,,2,0.563,4,3.13,-2
83,Recital 64,,6,0.530,5,3.03,1
267,Article 13 - Information to be provided where personal data,,8,0.529,6,2.71,2
60,Recital 48,,1,0.609,7,2.45,-6
75,Recital 57,,7,0.530,8,1.19,-1
90,Recital 68,*,4,0.533,9,0.82,-5


--- Answer 1 (baseline Top-3: [60, 82, 89] )
Yes, there is a right for a data subject to transmit or receive personal data concerning him or her [Recital 68]. However, this right should not be exercised against controllers processing personal data in the exercise of their public duties, specifically where processing is necessary for the performance of a task carried out in the public interest, the exercise of an official authority vested in the controller, or compliance with a legal obligation to which the controller is subject [Recital 68]. Additionally, this right does not obligate controllers to maintain or adopt processing systems that are technically compatible, and it must be without prejudice to the rights and freedoms of other data subjects [Recital 68].

--- Answer 2 (re-ranked Top-3: [285, 88, 276] )
Yes, you have the right to transmit your personal data to another controller without hindrance from the controller to which the data was provided, provided that:
* The processing

#### Case 2: Sub-processor liability (Keyword trap)

In [16]:
show_case(TEST_CASES[1])

Sub-processor liability  [Keyword trap]
Query: Who is liable if a sub-processor breaches its obligations?
Why it is hard: 'liable' and 'damage' pull in the general compensation rule (Art. 82, Recital 146); the specific sub-processor rule is a single sentence in Art. 28(4).
Critical chunk: #4 -> #1


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
304,Article 28 - Processor,*,4,0.630,1,2.58,3
441,Article 82 - Right to compensation and liability,,1,0.698,2,2.35,-1
197,Recital 146,,2,0.663,3,1.50,-1
196,Recital 146,,3,0.633,4,0.78,-1
442,Article 82 - Right to compensation and liability,,5,0.608,5,0.45,0
303,Article 28 - Processor,,6,0.593,6,0.29,0
128,Recital 95,,10,0.519,7,-2.00,3
300,Article 28 - Processor,,8,0.533,8,-3.95,0
305,Article 28 - Processor,,7,0.592,9,-4.63,-2


--- Answer 1 (baseline Top-3: [441, 197, 196] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [304, 441, 197] )
If another processor fails to fulfil its data protection obligations, the initial processor remains fully liable to the controller for the performance of that other processor's obligations (Article 28).


#### Case 3: Telling individuals about a breach (Exception / corner case)

In [17]:
show_case(TEST_CASES[2])

Telling individuals about a breach  [Exception / corner case]
Query: Do I always have to tell affected individuals about a data breach?
Why it is hard: Art. 34(3) lists three exceptions, but the list is split over two chunks: (a) encryption is in the chunk with the general duty, (b) later measures and (c) disproportionate effort are in the next chunk. Each pipeline only sees part of the list.
Critical chunk: #5 -> #2


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
115,Recital 85,,2,0.622,1,2.86,1
315,Article 34 - Communication of a personal data breach to the,*,5,0.604,2,1.01,3
117,Recital 86,,6,0.603,3,0.38,3
119,Recital 88,,3,0.614,4,0.01,-1
118,Recital 87,,7,0.572,5,-0.58,2
313,Article 33 - Notification of a personal data breach to the s,,4,0.605,6,-0.78,-2
314,Article 33 - Notification of a personal data breach to the s,,8,0.570,7,-1.02,1
316,Article 34 - Communication of a personal data breach to the,,1,0.630,8,-1.57,-7
422,Article 70 - Tasks of the Board,,9,0.516,9,-2.85,0


--- Answer 1 (baseline Top-3: [316, 115, 119] )
No, you do not always have to tell affected individuals. Communication to the data subject is not required if:
* The controller has taken subsequent measures ensuring that the high risk to the rights and freedoms of data subjects is no longer likely to materialise (Article 34(b)); or
* It would involve disproportionate effort, in which case a public communication or similar measure may be used instead to inform data subjects in an equally effective manner (Article 34(c)).

--- Answer 2 (re-ranked Top-3: [115, 315, 117] )
No, you do not always have to tell affected individuals. Communication to the data subject is required when the personal data breach is "likely to result in a high risk to the rights and freedoms of natural persons" (Article 34(1), Recital 86). Furthermore, communication is not required if the controller has implemented appropriate technical and organisational protection measures that render the affected personal data uni

#### Case 4: Dismissal of the DPO (Negation / role limit)

In [18]:
show_case(TEST_CASES[3])

Dismissal of the DPO  [Negation / role limit]
Query: Can a data protection officer be fired for doing their job?
Why it is hard: Articles 37, 38 and 39 are all about the DPO; only one chunk contains the prohibition 'shall not be dismissed or penalised', and the word 'fired' never appears in the text.
Critical chunk: #7 -> #2


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
328,Article 38 - Position of the data protection officer,,1,0.615,1,2.45,0
327,Article 38 - Position of the data protection officer,*,7,0.495,2,2.10,5
330,Article 39 - Tasks of the data protection officer,,3,0.528,3,1.79,0
326,Article 37 - Designation of the data protection officer,,2,0.541,4,0.72,-2
329,Article 39 - Tasks of the data protection officer,,4,0.514,5,0.67,-1
325,Article 37 - Designation of the data protection officer,,5,0.501,6,-0.38,-1
127,Recital 94,,10,0.454,7,-3.65,3
118,Recital 87,,6,0.498,8,-6.22,-2
115,Recital 85,,8,0.460,9,-6.50,-1


--- Answer 1 (baseline Top-3: [328, 326, 330] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [328, 327, 330] )
No, the data protection officer shall not be dismissed or penalised by the controller or the processor for performing his tasks (Article 38).


#### Case 5: Police and criminal investigations (Document-specific: scope exclusion)

In [19]:
show_case(TEST_CASES[4])

Police and criminal investigations  [Document-specific: scope exclusion]
Query: Does GDPR apply to the police investigating crimes?
Why it is hard: Chunks about criminal-conviction data (Art. 10) look relevant, but the answer is the scope exclusion in Art. 2(2)(d) / Recital 19 (that processing falls under Directive 2016/680).
Critical chunk: #4 -> #1


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
20,Recital 19,*,4,0.488,1,-8.39,3
230,Article 2 - Material scope,*,7,0.468,2,-9.19,5
23,Recital 19,,10,0.454,3,-10.11,7
260,Article 10 - Processing of personal data relating to crimina,,1,0.526,4,-10.39,-3
309,Article 30 - Records of processing activities,,2,0.504,5,-10.64,-3
318,Article 35 - Data protection impact assessment,,3,0.492,6,-10.68,-3
130,Recital 97,,9,0.455,7,-10.70,2
325,Article 37 - Designation of the data protection officer,,8,0.467,8,-10.83,0
250,Article 6 - Lawfulness of processing,,5,0.488,9,-11.11,-4


--- Answer 1 (baseline Top-3: [260, 309, 318] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [20, 230, 23] )
No, the GDPR does not apply to the processing of personal data by competent authorities for the purposes of the prevention, investigation, detection or prosecution of criminal offences (Article 2(2)(d), Recital 19).


#### Case 6: Processor must alert the controller (Document-specific: role-specific duty)

In [20]:
show_case(TEST_CASES[5])

Processor must alert the controller  [Document-specific: role-specific duty]
Query: Must a processor tell the controller about a breach?
Why it is hard: Chunks about processors in general (Art. 28, 29) outrank the one sentence in Art. 33(2) that gives the processor its own breach duty.
Critical chunk: #6 -> #1


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
313,Article 33 - Notification of a personal data breach to the s,*,6,0.490,1,4.75,5
128,Recital 95,,9,0.474,2,1.94,7
196,Recital 146,,10,0.473,3,1.86,7
115,Recital 85,,2,0.531,4,1.70,-2
197,Recital 146,,7,0.480,5,1.10,2
303,Article 28 - Processor,,3,0.525,6,0.86,-3
112,Recital 82,,8,0.478,7,0.22,1
306,Article 29 - Processing under the authority of the controlle,,1,0.537,8,-0.02,-7
314,Article 33 - Notification of a personal data breach to the s,,4,0.500,9,-0.25,-5


--- Answer 1 (baseline Top-3: [306, 115, 303] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [313, 128, 196] )
Yes, the processor shall notify the controller without undue delay after becoming aware of a personal data breach (Article 33).


#### Case 7: Secrecy of the supervisory authority (Buried chunk (rank 10))

In [21]:
show_case(TEST_CASES[6])

Secrecy of the supervisory authority  [Buried chunk (rank 10)]
Query: Does the regulator have to keep secrets?
Why it is hard: Art. 90 'Obligations of secrecy' (about professional secrecy of controllers) is the obvious lexical match; the duty of the authority's own staff is in Art. 54(2), ranked last by the bi-encoder.
Critical chunk: #10 -> #2


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
219,Recital 164,,3,0.436,1,-3.00,2
375,Article 54 - Rules on the establishment of the supervisory a,*,10,0.381,2,-4.86,8
274,Article 14 - Information to be provided where personal data,,2,0.459,3,-5.04,-1
459,Article 90 - Obligations of secrecy,,1,0.493,4,-6.42,-3
385,Article 58 - Powers,,8,0.386,5,-10.03,3
434,Article 76 - Confidentiality,,4,0.426,6,-10.26,-2
370,Article 51 - Supervisory authority,,5,0.393,7,-10.35,-2
340,Article 42 - Certification,,6,0.390,8,-10.50,-2
35,Recital 29,,9,0.383,9,-10.54,0


--- Answer 1 (baseline Top-3: [459, 274, 219] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [219, 375, 274] )
Yes, the members and staff of each supervisory authority are subject to a duty of professional secrecy, both during and after their term of office, regarding any confidential information they learn while performing their tasks or exercising their powers (Article 54).


#### Case 8: Regulator ignores a complaint (Limitation: re-ranking hurts)

In [22]:
show_case(TEST_CASES[7])

Regulator ignores a complaint  [Limitation: re-ranking hurts]
Query: What can I do if the regulator ignores my complaint?
Why it is hard: Counter-example: the cross-encoder prefers recitals that talk about complaints in general and pushes the Art. 78(2) remedy further down.
Critical chunk: #4 -> #8


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
192,Recital 143,,6,0.356,1,-3.53,5
166,Recital 125,,3,0.371,2,-3.85,1
174,Recital 130,,2,0.382,3,-4.63,-1
191,Recital 143,,5,0.359,4,-6.21,1
437,Article 79 - Right to an effective judicial remedy against a,,10,0.348,5,-6.75,5
412,Article 65 - Dispute resolution by the Board,,1,0.392,6,-7.12,-5
393,Article 60 - Cooperation between the lead supervisory author,,7,0.351,7,-7.55,0
436,Article 78 - Right to an effective judicial remedy against a,*,4,0.366,8,-8.23,-4
190,Recital 143,,9,0.348,9,-9.17,0


--- Answer 1 (baseline Top-3: [412, 174, 166] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [192, 166, 174] )
The provided context does not answer this.


#### Case 9: Switching provider (Limitation: recall ceiling)

In [23]:
show_case(TEST_CASES[8])

Switching provider  [Limitation: recall ceiling]
Query: Can I transfer my data to another company that offers the same service?
Why it is hard: Counter-example: the keyword trap is so strong that Art. 20 is not in the bi-encoder Top-10 at all, so the re-ranker never sees it. Re-ranking can only reorder what stage 1 retrieved.
Critical chunk: not in Top-10 -> not in Top-10


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
348,Article 44 - General principle for transfers,,6,0.435,1,-1.36,5
135,Recital 101,,5,0.436,2,-1.96,3
150,Recital 112,,9,0.420,3,-2.63,6
143,Recital 107,,8,0.420,4,-3.52,4
366,Article 49 - Derogations for specific situations,,4,0.437,5,-3.81,-1
355,Article 46 - Transfers subject to appropriate safeguards,,7,0.434,6,-3.98,1
148,Recital 111,,10,0.413,7,-4.14,3
60,Recital 48,,2,0.447,8,-4.67,-6
136,Recital 102,,1,0.490,9,-4.69,-8


--- Answer 1 (baseline Top-3: [136, 60, 276] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [348, 135, 150] )
The provided context does not answer this.


## 9. Latency profiling

* Retrieval: every showcase query is run `N_REPEATS` times after a warm-up pass; the **median** is the main
  number because CPU timings on a laptop have occasional slow outliers.
* LLM: one timed API call per answer (rate-limit waits excluded). Network time dominates and varies a lot, so the
  median is reported.
* One-time costs (model loading, indexing) are reported separately because they are paid once, not per query.

In [24]:
import torch

print(f"Hardware: {platform.processor()} | CPU only | torch threads = {torch.get_num_threads()}")
print(f"One-time cost: embedding and indexing {len(chunks)} chunks took {index_seconds:.1f} s")

N_REPEATS = 30

for case in TEST_CASES:  # warm-up
    run_pipelines(case.query, with_llm=False)

timings = []
for case in TEST_CASES:
    for _ in range(N_REPEATS):
        run = run_pipelines(case.query, with_llm=False)
        timings.append({"bi": run.retrieval_ms, "ce": run.rerank_ms})
timings = pd.DataFrame(timings)

stages = pd.DataFrame({
    "Pipeline 1: bi-encoder retrieval (Top-10)": timings["bi"],
    "Pipeline 2: bi-encoder + cross-encoder": timings["bi"] + timings["ce"],
    "   of which cross-encoder (10 pairs)": timings["ce"],
})
retrieval_table = stages.describe(percentiles=[0.5, 0.95]).T[["50%", "mean", "95%"]]
retrieval_table.columns = ["median (ms)", "mean (ms)", "p95 (ms)"]
print(f"{len(timings)} timed runs per pipeline")
retrieval_table.round(1)

Hardware: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel | CPU only | torch threads = 8
One-time cost: embedding and indexing 471 chunks took 9.2 s


270 timed runs per pipeline


,median (ms),mean (ms),p95 (ms)
Pipeline 1: bi-encoder retrieval (Top-10),8.4,12.6,37.1
Pipeline 2: bi-encoder + cross-encoder,279.7,375.7,954.5
of which cross-encoder (10 pairs),271.7,363.0,916.3


In [25]:
llm_rows = [{"case": name, "LLM call P1 (ms)": run.llm_1_ms, "LLM call P2 (ms)": run.llm_2_ms}
            for name, run in runs.items() if run.llm_1_ms is not None]

if llm_rows:
    llm_latency = pd.DataFrame(llm_rows).set_index("case")
    display(llm_latency.round(0))

    retrieval_median = retrieval_table["median (ms)"]
    llm_median = llm_latency.median()
    end_to_end = pd.DataFrame({
        "retrieval (median ms)": [retrieval_median.iloc[0], retrieval_median.iloc[1]],
        "LLM call (median ms)": [llm_median["LLM call P1 (ms)"], llm_median["LLM call P2 (ms)"]],
    }, index=["Pipeline 1", "Pipeline 2"])
    end_to_end["end-to-end (ms)"] = end_to_end.sum(axis=1)
    display(end_to_end.round(0))
else:
    print("LLM latency not measured (no API key and no cached answers).")

,LLM call P1 (ms),LLM call P2 (ms)
case,,
Data portability vs. international transfers,26788.0,34063.0
Sub-processor liability,64493.0,49307.0
Telling individuals about a breach,48556.0,45297.0
Dismissal of the DPO,60517.0,54281.0
Police and criminal investigations,175496.0,40374.0
Processor must alert the controller,79189.0,38713.0
Secrecy of the supervisory authority,43316.0,85913.0
Regulator ignores a complaint,43107.0,25003.0
Switching provider,27428.0,32582.0


,retrieval (median ms),LLM call (median ms),end-to-end (ms)
Pipeline 1,8.0,48556.0,48565.0
Pipeline 2,280.0,40374.0,40654.0


## 10. Scaling experiment: why not cross-encode the whole document?

Here the cross-encoder is run against **every** chunk for one query and compared to the bi-encoder search.
The bi-encoder's chunk embeddings are computed once offline; at query time it only embeds the query and does one
matrix-vector product. The cross-encoder has no offline part: every (query, chunk) pair is a full transformer pass.

In [26]:
query = TEST_CASES[0].query
pairs = [(query, text) for text in chunk_texts]

start = time.perf_counter()
store.search(query, RETRIEVE_K)
bi_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
cross_encoder.predict(pairs, batch_size=32)
ce_full_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
cross_encoder.predict(pairs[:RETRIEVE_K])
ce_top10_ms = (time.perf_counter() - start) * 1000

ce_per_pair_ms = ce_full_ms / len(pairs)
scaling = pd.DataFrame({
    "corpus size (chunks)": [len(chunks), 100_000, 10_000_000],
    "cross-encoder on every chunk (s)": [ce_full_ms / 1000, ce_per_pair_ms * 100_000 / 1000,
                                         ce_per_pair_ms * 10_000_000 / 1000],
}).set_index("corpus size (chunks)")

print(f"Bi-encoder search over {len(chunks)} chunks : {bi_ms:8.1f} ms")
print(f"Cross-encoder on Top-{RETRIEVE_K} only          : {ce_top10_ms:8.1f} ms")
print(f"Cross-encoder on all {len(chunks)} chunks      : {ce_full_ms:8.1f} ms  ({ce_per_pair_ms:.2f} ms per pair)")
scaling.round(1)

Bi-encoder search over 471 chunks :      8.7 ms
Cross-encoder on Top-10 only          :    259.1 ms
Cross-encoder on all 471 chunks      :  10732.5 ms  (22.79 ms per pair)


,cross-encoder on every chunk (s)
corpus size (chunks),
471,10.7
100000,2278.7
10000000,227866.1
